## Evaluate models trained with tsimcne

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split

from dual_ifm.tsimcne import metrics, models
from dual_ifm.tsimcne.eval import load_eval
from dual_ifm.utils import plot
from dual_ifm.utils.datasets import load_dataset_all

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)
    
plot.set_rc_params(kind='paper', notebook_dpi=120)

### Load results

In [ ]:
experiment_name = 'tsimcne_resnet18_cifar10_32_100_float16'
dataset_eval = 'cifar10'
image_size = (32, 32)
sample_size = None
dataset_dir = './datasets'

project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints', experiment_name)
X, y, stats, wandb_id = load_eval(checkpoints_dir, experiment_name)
print('Logged in W&B with id = ', wandb_id)

### Learning curve

In [ ]:
loss_label = 'InfoNCE Cauchy' if 'tsimcne' in experiment_name else 'NT-Xent'

fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)

ax.plot(stats['loss_train'], '-o', markersize=3)
ax.set_title(loss_label)
ax.set_xlabel('Epoch')
ax.set_ylabel('Average Loss')

plt.tight_layout()
# fig.savefig(plot_file)

### Load dataset

In [ ]:
# Load dataset mapping
if dataset_eval == 'eyepacs':    
    feature_name = ['age', 'gender', 'ethnicity', 'dr', 'dme', 'camera']
elif dataset_eval == 'areds':
    feature_name = ['age', 'gender', 'hbp', 'amd', 'diabetes', 'smoking']
elif dataset_eval in ['aptos', 'deepdrid', 'idrid', 'messidor']:
    feature_name = ['dr']
elif dataset_eval in ['glaucoma', 'papila']:
    feature_name = ['glaucoma']
elif dataset_eval in ['fives']:
    feature_name = ['disease']
else:
    feature_name = ['Classes']

dataset, mappings = load_dataset_all(dataset_dir=dataset_dir, dataset_name=dataset_eval, transform=None, image_size=image_size, feature_name=feature_name, drop_nan=False, sample_size=sample_size)

if dataset_eval not in ['eyepacs', 'areds']:
    mappings = [mappings]
else:
    name2label = {'dr': 'diabetic retinopathy', 'dme': 'diabetic macular edema', 'hbp': 'high blood pressure'}
    feature_name = [name2label.get(feature, feature) for feature in feature_name]

y = dataset.labels if dataset_eval not in ['cifar10', 'imagenette'] else y
y = np.expand_dims(y, -1) if dataset_eval not in ['eyepacs', 'areds'] else y

### Evaluate with KNN

In [ ]:
# Evaluate
X_train, X_test, y_train, y_test = train_test_split(X, y)

print('KNN accuracy / R2')
for i in range(y.shape[1]):
    # Drop nans
    X_train_, y_train_, _ = plot.drop_nans(X_train, y_train[:, i])
    X_test_, y_test_, _ = plot.drop_nans(X_test, y_test[:, i])

    if feature_name[i] != 'age':
        metric = 100 * metrics.knn_acc(X_train_, X_test_, y_train_, y_test_)
        print(f'{feature_name[i].capitalize()} = {metric:.2f}%')
    else:
        metric = metrics.knn_reg(X_train_, X_test_, y_train_, y_test_)
        print(f'{feature_name[i].capitalize()} = {metric:.3f}')

### Visualize 2D embeddings

In [ ]:
if dataset_eval in ['eyepacs', 'areds']:
    n_subplots = (2, 3)
    fig_width = 'full'
    fig_height_ratio = 0.6
    if dataset_eval == 'eyepacs':
        imbalanced = [False, False, True, True, True, False]
        categorical = [False] + 5*[True]
    else:
        imbalanced = [False, False, True, False, True, False]
        categorical = [False, True, True, False, True, True]
    feature_label = feature_name
else:
    n_subplots = (1, 1)
    fig_width = 'col'
    fig_height_ratio = 1
    imbalanced = [False]
    categorical = [True]
    feature_label = [dataset_eval + ': ' + f for f in feature_name]

plot.plot_embeddings(X, y, mappings, None, n_subplots, fig_width, fig_height_ratio, feature_label, imbalanced, categorical)

### Visualize weight distribution

In [ ]:
# Load model
model = models.CNNwithProjector(img_size=image_size, backbone='resnet18', weights=None)
checkpoint_file = checkpoints_dir.joinpath('f{experiment_name}.pt')

# Load checkpoint if it exists
if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location=torch.device('cpu'), weights_only=False)

    if 'tsimcne' in experiment_name:
        start_stage = checkpoint['stage']
        if start_stage > 0:
            model.mutate_projector()
    model.load_state_dict(checkpoint['state_dict'])

fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)
for name, parameters in model.named_parameters():
    if 'conv' in name:
        params = parameters.detach().cpu().numpy().flatten()
        sns.kdeplot(x=params, ax=ax)

# ax.set_xlim([-2, 2])
# ax.set_ylim([1e-10, 1e8])
# ax.set_yscale('log')
ax.set_title('Model weight distribution')
ax.set_xlabel('Weight Values')
plt.tight_layout()